# FABLE: Fast Approximate Block-Encodings of Dense Matrices

*Block encoding* embeds an arbitrary matrix $A$ into a larger unitary so that quantum
linear-algebra primitives (Hamiltonian simulation, the quantum singular value transformation,
linear-system solvers) can act on it. A unitary $U$ is an $(\alpha, m, \epsilon)$-block-encoding
of a matrix $A$ (acting on $n$ data qubits) if

$$
\left\| A - \alpha\,\bigl(\langle 0|^{\otimes m} \otimes I_{2^n}\bigr)\,U\,\bigl(|0\rangle^{\otimes m} \otimes I_{2^n}\bigr) \right\| \le \epsilon ,
$$

i.e. $A/\alpha$ sits in the top-left block of $U$, addressed by setting the $m$ block (ancilla)
qubits to $|0\rangle$. Here $\alpha$ is the **subnormalization factor**, $m$ the **block size**,
and $\epsilon$ the encoding error.

Sparse-oracle methods are the right tool when $A$ has few nonzeros per row and column (see the
[sparse-oracle block-encoding notebook](https://github.com/Classiq/classiq-library/blob/main/algorithms/quantum_primitives/sparse_oracle_and_fable/block_encoding_sparse_oracles_and_fable.ipynb)). For a *dense* matrix a different strategy is
preferable. **FABLE** (Fast Approximate BLock-Encodings) [[1](#FABLE)] writes every entry directly
into the circuit as a uniformly-controlled rotation, then optionally compresses it by discarding
small rotation angles in the Walsh-Hadamard domain. It is exposed through
`BlockEncoding.from_matrix(mat, method="fable", ...)`.

In this notebook we:

1. Describe the FABLE construction (amplitude oracle, Walsh-Hadamard compression).
2. Build an exact dense block encoding and verify it.
3. Compress a smooth matrix and watch the depth and CX count fall as the accuracy degrades gracefully.

## 1. Setup

We import the `BlockEncoding` class and define two helpers: one that checks a reconstructed
matrix against the expected one up to a global phase (the `to_matrix()` method recovers $A$ only
up to a physically unobservable overall phase), and one that synthesizes a block encoding and
reports its transpiled width, depth, and CX count.

In [1]:
import logging

import numpy as np

from classiq import *
from classiq.applications.block_encoding import BlockEncoding

# Silence the per-job "Submitting state-vector job to simulator" log line.
logging.getLogger("classiq.execution.functions.util._logging").setLevel(logging.WARNING)


def assert_close_up_to_global_phase(actual, expected, atol=1e-5):
    # Assert two matrices are equal up to a single (unknown) global phase.
    actual = np.asarray(actual, dtype=complex)
    expected = np.asarray(expected, dtype=complex)
    overlap = np.vdot(expected, actual)
    assert abs(overlap) > atol, "matrices are (nearly) orthogonal; phase is undefined"
    phase = overlap / abs(overlap)
    np.testing.assert_allclose(actual, phase * expected, atol=atol)
    print("reconstruction confirmed")


def block_encoding_metrics(block_encoding):
    # Synthesize the block-encoding unitary and read off transpiled width, depth, CX count.
    @qfunc
    def main(data: Output[QNum], block: Output[QNum]) -> None:
        allocate(block_encoding.data_size, data)
        allocate(block_encoding.block_size, block)
        block_encoding.unitary(data, block)

    metrics = get_transpiled_circuit_metrics(synthesize(main))
    return metrics.width, metrics.depth, metrics.count_ops.get("cx", 0)

## 2. The FABLE algorithm

Throughout, $N = 2^n$ (non-square or non-power-of-two inputs are zero-padded to the next $2^n$) and the entries are rescaled to
$\hat a_{ij} = a_{ij}/\max_{kl}|a_{kl}|$ so that $|\hat a_{ij}| \le 1$, the condition under which
the block encoding exists.

**The amplitude oracle.** FABLE loads the data through an amplitude oracle $O_A$ that rotates an
amplitude ancilla so its $|0\rangle$-amplitude equals the addressed, rescaled entry. With dense
access the row and column are supplied directly through two $n$-qubit registers $|i\rangle,|j\rangle$,
so

$$
O_A\,|0\rangle|i\rangle|j\rangle = \Bigl(\hat a_{ij}\,|0\rangle + \sqrt{1-|\hat a_{ij}|^{2}}\,|1\rangle\Bigr)\,|i\rangle|j\rangle .
$$

(The same amplitude-oracle idea underlies the sparse-oracle methods, where the row is addressed
indirectly through a sparse index instead of a full register; see the
[sparse-oracle notebook](https://github.com/Classiq/classiq-library/blob/main/algorithms/quantum_primitives/sparse_oracle_and_fable/block_encoding_sparse_oracles_and_fable.ipynb).)

Wrapping $O_A$ between Hadamards on the row quantum variable and a data-index `SWAP` gives the
full FABLE unitary

$$
U_A = \bigl(I \otimes H^{\otimes n} \otimes I_n\bigr)\,\bigl(I \otimes \mathrm{SWAP}\bigr)\,O_A\,\bigl(I \otimes H^{\otimes n} \otimes I_n\bigr),
$$

which satisfies $\langle 0|\langle 0|^{\otimes n}\langle i|\,U_A\,|0\rangle|0\rangle^{\otimes n}|j\rangle = \hat a_{ij}/N$.
Thus $U_A$ is an $(N,\,n+1)$-block-encoding of the rescaled matrix. Carrying the rescaling back
gives subnormalization $\alpha = \max_{kl}|a_{kl}|\cdot N$, block size $n+1$ (the $n$-qubit
row/index quantum variable plus the amplitude ancilla), and data size $n$. This $\alpha$ is chosen so that `to_matrix()` returns $A$ itself (up to global phase).

**Encoding the entries as angles.** For a real matrix $O_A$ acts on the ancilla as a single $R_y$
with

$$
\theta_{ij} = \arccos(\hat a_{ij}),\qquad
R_y(2\theta_{ij})\,|0\rangle = \hat a_{ij}\,|0\rangle + \sqrt{1-\hat a_{ij}^{2}}\,|1\rangle .
$$

A complex entry $\hat a_{ij} = |\hat a_{ij}|\,e^{i\gamma_{ij}}$ adds an $R_z$ pass,

$$
\theta_{ij} = \arccos(|\hat a_{ij}|),\qquad \phi_{ij} = -\gamma_{ij},\qquad
R_z(2\phi_{ij})\,R_y(2\theta_{ij})\,|0\rangle ,
$$

so the $R_y$ pass fixes the magnitude and the $R_z$ pass fixes the phase.

**Walsh-Hadamard transform motivation.** A naive $O_A$ would use one multi-controlled rotation per
entry, that is, $N^2$ of them, each costing $O(N^2)$ elementary gates, to give $O(N^4)$ overall.
Instead FABLE realizes $O_A$ as a *uniformly-controlled rotation* (UCR) (also referred to as a
multiplexed rotation), a length-$N^2$ sequence alternating a single-qubit $R_y$ on the ancilla
with a single CNOT whose control is fixed by the bit in which consecutive Gray-code indices
differ. The circuit angles $\hat{\boldsymbol{\theta}}$ are not the entry angles
$\boldsymbol{\theta}$ but their solution of the linear system

$$
\bigl(\hat H^{\otimes 2n}\, P_G\bigr)\,\hat{\boldsymbol{\theta}} = \boldsymbol{\theta},
\qquad \hat H = \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix},
$$

with $P_G$ the binary-to-Gray-code permutation. This is solved classically in $O(N^2\log N^2)$ by a
fast Walsh-Hadamard transform, and it brings the quantum gate count down to $O(N^2)$ ($N^2$ CNOTs
and $N^2$ rotations, with a prefactor $2$ for real data, $4$ for complex). This is the optimal
asymptotic cost for unstructured dense data.

**Compression.** Passing `angle_threshold` $= \delta_c > 0$ drops every circuit rotation with
$|\hat\theta_i| \le \delta_c$. As a consequence, the CNOTs left behind in each consecutive run are
then merged by a parity rule, since $\mathrm{CX}^2 = I$ it keeps only one CNOT per control that
occurs an odd number of times. Because the thresholding happens on $\hat{\boldsymbol\theta}$ after
the Walsh-Hadamard transform, FABLE is efficient precisely for matrices that are sparse in the
Walsh-Hadamard domain, which need not be sparse in the usual sense. The magnitude ($R_y$) and phase
($R_z$) passes are compressed separately, with the phase mask gated by the magnitude mask (a phase
is only meaningful on a nonzero magnitude, so if a position's magnitude is dropped its phase is
dropped too, but not vice versa). The resulting error is bounded (not tight) to first order (Thm. 2
of [[1](#FABLE)]) by

$$
\bigl\| A - \alpha\,\tilde A \bigr\|_2 \le N^3\,\delta_c + O(\delta_c^{3}),
$$

so $U_A$ becomes an $(\alpha,\, n+1,\, \epsilon)$-block-encoding with $\epsilon = N^3\,\delta_c$.
This is the value reported in the constructor's `epsilon` field.

## 3. Exact encoding

We block-encode a dense $4\times4$ matrix. With the default `angle_threshold=0.0` the encoding is
exact, so `to_matrix()` recovers $A$ (up to global phase) and the subnormalization is
$\alpha = \max_{ij}|A_{ij}|\cdot N = 0.7\cdot 4 = 2.8$.

In [2]:
A = np.array(
    [
        [0.5, 0.3, 0.1, 0.2],
        [0.4, 0.6, 0.2, 0.3],
        [0.2, 0.1, 0.7, 0.4],
        [0.1, 0.2, 0.3, 0.5],
    ]
)

be_fable = BlockEncoding.from_matrix(A, method="fable")
print(
    f"alpha = {be_fable.alpha}, block_size = {be_fable.block_size}, "
    f"data_size = {be_fable.data_size}, epsilon = {be_fable.epsilon}"
)
assert_close_up_to_global_phase(be_fable.to_matrix(), A)

alpha = 2.8, block_size = 3, data_size = 2, epsilon = 0.0


reconstruction confirmed


To inspect the circuit we wrap the block encoding in a `main` qfunc, synthesize it, and show
the resulting quantum program.

In [3]:
@qfunc
def main(data: Output[QNum], block: Output[QNum]) -> None:
    allocate(be_fable.data_size, data)
    allocate(be_fable.block_size, block)
    be_fable.unitary(data, block)


qprog = synthesize(main)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/3M8ff0GLWhaE3iqDplFdxRRCf2s


## 4. Approximate (compressed) encoding

Passing `angle_threshold > 0` drops small rotations in the Walsh-Hadamard domain, trading accuracy
for a shallower circuit. On a smooth (hence Walsh-sparse) dense matrix the savings in CX count and
depth are substantial, while the reconstruction error stays bounded by
$\epsilon = N^3\cdot\texttt{angle\_threshold}$ (the value reported in the constructor's `epsilon` field) is a loose
first-order bound that the measured error below comfortably beats. We synthesize each encoding and
read off the transpiled gate counts.

In [4]:
# A smooth (therefore Walsh-sparse) dense 8x8 matrix compresses well.
grid = np.linspace(0, 1, 8)
A_smooth = np.exp(-((grid[:, None] - grid[None, :]) ** 2) / 0.1)
A_smooth /= np.max(np.abs(A_smooth))

print(f"{'threshold':>9}{'depth':>7}{'cx':>5}{'max|recon-A|':>14}")
for threshold in [0.0, 0.05, 0.1, 0.2]:
    be_t = BlockEncoding.from_matrix(
        A_smooth, method="fable", angle_threshold=threshold
    )
    recon = be_t.to_matrix()
    overlap = np.vdot(A_smooth, recon)
    phase = overlap / abs(overlap)
    error = np.max(np.abs(recon - phase * A_smooth))
    _, depth, cx = block_encoding_metrics(be_t)
    print(f"{threshold:>9}{depth:>7}{cx:>5}{error:>14.4f}")

threshold  depth   cx  max|recon-A|


      0.0    100   73        0.0000


     0.05     80   65        0.1844


      0.1     68   57        0.3183


      0.2     20   21        0.5636


## References

<a id='FABLE'>[1]</a>: [D. Camps, R. Van Beeumen, *FABLE: Fast Approximate Quantum Circuits for Block-Encodings*, IEEE QCE 2022 (arXiv:2205.00081)](https://arxiv.org/abs/2205.00081)